# 03.6 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/03-classification/06-confusion-matrix-metrics/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

cust20 = pd.read_csv(data_path("shopwave_customers_small.csv"))
X20 = cust20[["visits_last_month", "days_since_last"]]
y20 = cust20["churned"]
cust20

,customer_id,visits_last_month,items_last_month,days_since_last,country,spend_next_month,churned
0,C001,2,2,55,UK,27.61,1
1,C002,11,8,44,FR,220.42,0
2,C003,10,6,22,DE,176.21,0
3,C004,7,6,58,UK,127.58,1
4,C005,7,5,25,FR,150.95,1
5,C006,13,7,20,FR,250.03,0
6,C007,2,2,54,UK,47.33,1
7,C008,10,5,22,UK,196.92,0
8,C009,3,3,5,DE,94.82,1
9,C010,2,1,28,UK,45.05,1


## Exercise 1

Rule `> 40` flags C001 (55), C002 (44), C004 (58), C007 (54), C011 (47), C015 (41): six customers. Of those, churned: C001, C004, C007, C011 → **TP = 4, FP = 2**. Churners not flagged: C005, C009, C010, C018, C019 → **FN = 5**. **TN = 20 − 4 − 2 − 5 = 9**.

Accuracy = 13/20 = 0.65 (unchanged). Precision = 4/6 = **0.667 (up)**. Recall = 4/9 = **0.444 (down)**. F1 = 8/(8+2+5) = 0.533.

Stricter rule → fewer flags → each flag more trustworthy (precision up) but more churners missed (recall down). That trade-off is the subject of 03.7.

In [2]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
p40 = (cust20["days_since_last"] > 40).astype(int)
print(confusion_matrix(cust20["churned"], p40))
for f in (accuracy_score, precision_score, recall_score, f1_score):
    print(f"{f.__name__:<16} {f(cust20['churned'], p40):.3f}")

[[9 2]
 [5 4]]
accuracy_score   0.650
precision_score  0.667
recall_score     0.444
f1_score         0.533


## Exercise 2

Flagged 120, of which 80 fraud → **TP = 80, FP = 40**. Frauds in total 200 → **FN = 120**. **TN = 10,000 − 80 − 40 − 120 = 9,760**.

- Precision = 80/120 = **0.667**
- Recall = 80/200 = **0.400**
- F1 = 160/(160 + 40 + 120) = **0.500**
- Accuracy = 9,840/10,000 = **0.984**

98.4% accurate and it misses six frauds in ten.

In [3]:
TP, FP, FN, TN = 80, 40, 120, 9760
print("precision", TP/(TP+FP), " recall", TP/(TP+FN), " f1", 2*TP/(2*TP+FP+FN), " accuracy", (TP+TN)/10000)

precision 0.6666666666666666  recall 0.4  f1 0.5  accuracy 0.984


## Exercise 3

In [4]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd, numpy as np
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail
from sklearn.model_selection import train_test_split

xlsx = ensure_online_retail()
csv = xlsx.with_name("online_retail_2010_2011.csv")          # fast cache written by lesson 01.3
raw = pd.read_csv(csv, parse_dates=["InvoiceDate"]) if csv.exists() else pd.read_excel(xlsx, sheet_name="Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

3,317 customers, churn rate 43.6%, train 2,487 / test 830


In [5]:
rows = []
for t in [-1, 15, 30, 60, 90, 120, 150]:
    p = (X_test["days_since_last"] > t).astype(int)
    rows.append(dict(t=t, flagged=int(p.sum()), precision=precision_score(y_test, p),
                     recall=recall_score(y_test, p), f1=f1_score(y_test, p)))
pd.DataFrame(rows).set_index("t").round(3)

,flagged,precision,recall,f1
t,,,,
-1,830,0.436,1.000,0.607
15,708,0.479,0.936,0.634
30,606,0.507,0.848,0.634
60,473,0.533,0.696,0.604
90,344,0.564,0.536,0.550
120,243,0.646,0.434,0.519
150,189,0.677,0.354,0.465


Recall rises all the way to 1.0 as the threshold falls (t = −1 flags everyone, misses nobody). F1 peaks at a low but non-zero threshold and then **falls back** at "flag everyone", because precision collapses to the churn rate. They differ because recall ignores false alarms and F1 does not. Note how low the best threshold is here: most churners stopped ordering only recently, so a 90-day rule misses half of them.

## Exercise 4

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)
proba_test = logreg.predict_proba(X_test)[:, 1]          # P(churn) for each test customer
pred_test = logreg.predict(X_test)                        # same as (proba_test >= 0.5)

from sklearn.metrics import classification_report
print("stay class: precision", round(precision_score(y_test, pred_test, pos_label=0), 3),
      " recall", round(recall_score(y_test, pred_test, pos_label=0), 3))
print(classification_report(y_test, pred_test, target_names=["stay", "churn"]))

stay class: precision 0.721  recall 0.69
              precision    recall  f1-score   support

        stay       0.72      0.69      0.71       468
       churn       0.62      0.65      0.64       362

    accuracy                           0.67       830
   macro avg       0.67      0.67      0.67       830
weighted avg       0.68      0.67      0.68       830



The `stay` row of the report matches. The report shows both because "precision" and "recall" are defined *per class*: a model can be excellent at confirming stayers and poor at catching churners, and one row alone would hide that.